# Thickness of one current sheet, five ways

One notebook for the three that did this run by run (`SingleCSAnalysisBeta0.0625`, `SingleCSAnalysisBeta1`,
`SingleCSAnalysisBeta16`) and the thickness part of `SingleCSAnalysisBeta0.0625_HarrisFit`, all four of which
it replaces. For each case in `CASES`: $J^2$ at one time (from B filtered as the case says), the current
sheets above the 0.8 quantile, and for the chosen sheet the cut along the steepest descent of $J^2$ and five thicknesses:

1. diameter of the largest circle inside the detected sheet;
2. width of the detected sheet along the cut;
3. FWHM of a Gaussian fitted to $J^2$ along the cut (whole-cell samples);
4. width along the cut of the region around the sheet where $J^2 \geq$ 0.005 of the maximum of $J^2$;
5. the same with $J^2 \geq$ 0.1 of $J^2$ at the sheet.

Lengths are in $d_p$; the table of all thicknesses is saved as `thicknesses.csv`.

In [ ]:
import csv

import numpy as np
import matplotlib.pyplot as plt

from pta import config
from pta.fields import current_density
from pta.filtering import iterative_filtering1D, iterative_filtering3D
from pta.plotting import LINE_COLOURS, mark_cells, outline_mask, save_figure, show_field, use_style
from pta.sheets import (descent_direction, fwhm_nearest_samples, gaussian, get_max_coords, grow_region, line_profile,
                        quantile_mask, thickness_m1, width_along)

use_style()
RESULTS = config.results_dir("single_sheet", "SingleCSAnalysis")

In [ ]:
SNAPSHOT_TIME = 40

# One case per run and filter; sheet is the cell of the chosen J^2 maximum (picked by eye from the maxima map).
# filter: None, "iterative_3d" (iterative_filtering3D from IMF ITERATIVE_INDEX, Xi = ITERATIVE_3D_XI) or
# "iterative_per_component" (iterative_filtering1D of Bx, By and Bz - 1, as in the former HarrisFit notebook).
# cut_half_length, comparison_half_length: the cuts shown (cells); zoom_half_box: the zoom on the regions of
# methods 4 and 5 (cells); region_smoothing: the disk (cells) that opens and closes those regions.
CASES = {
    "beta_p0.0625": dict(beta_p=0.0625, filter="iterative_3d", sheet=(3731, 2294), cut_half_length=60,
                         comparison_half_length=60, zoom_half_box=400, region_smoothing=2),
    "beta_p1": dict(beta_p=1, filter=None, sheet=(3732, 2228), cut_half_length=70,
                    comparison_half_length=70, zoom_half_box=400, region_smoothing=2),
    "beta_p16": dict(beta_p=16, filter=None, sheet=(3708, 2284), cut_half_length=40,
                     comparison_half_length=150, zoom_half_box=500, region_smoothing=5),
    "beta_p0.0625_per_component": dict(beta_p=0.0625, filter="iterative_per_component", sheet=(3731, 2294),
                                       cut_half_length=60, comparison_half_length=60, zoom_half_box=400,
                                       region_smoothing=2),
}
ITERATIVE_INDEX = 1
ITERATIVE_3D_XI = 2.0

# ---- the detected sheets ----
SHEET_QUANTILE = 0.8                  # J^2 above this quantile
SMOOTHING_RADIUS_CELLS = 5
MIN_AREA_CELLS = 300

# ---- thickness methods ----
FWHM_HALF_LENGTH_CELLS = 60           # method 3: samples at k = -60..60 cells along the cut
METHOD4_FRACTION_OF_MAX = 0.005       # method 4: J^2 >= this fraction of the maximum of J^2
METHOD5_FRACTION_OF_PEAK = 0.1        # method 5: J^2 >= this fraction of J^2 at the sheet
CUT_ZOOM_HALF_BOX = 500               # zoom on the cut (cells)

In [ ]:
def J_squared_at(case):
    """J^2 of one case at SNAPSHOT_TIME, from B filtered as the case says, and the grid spacing."""
    run = config.run("beta_scan", beta_p=case["beta_p"])
    Bx, By, Bz = run.load_many(["Bx", "By", "Bz"], SNAPSHOT_TIME)
    if case["filter"] == "iterative_3d":
        Bx, By, Bz = iterative_filtering3D(Bx, By, Bz, ITERATIVE_INDEX, Xi=ITERATIVE_3D_XI)
    elif case["filter"] == "iterative_per_component":
        Bx, By, Bz = (iterative_filtering1D(component, ITERATIVE_INDEX) for component in (Bx, By, Bz - 1))
    Jx, Jy, Jz = current_density(Bx, By, Bz, run.dx, run.dy)
    return Jx**2 + Jy**2 + Jz**2, run.dx, run.dy


def zoom(ax, field, centre, half_box, dx, dy, label=None):
    """The part of the field within half_box cells of the centre, in d_p (positions as in the whole box)."""
    i0, j0 = max(centre[0] - half_box, 0), max(centre[1] - half_box, 0)
    show_field(ax, field[i0:centre[0] + half_box, j0:centre[1] + half_box], dx, dy, label=label,
               corner=(i0 * dx, j0 * dy))


def cell_centre(cell, dx, dy):
    """Position in d_p of the centre of a cell."""
    return (cell[0] + 0.5) * dx, (cell[1] + 0.5) * dy

In [ ]:
def analyse_case(name, case):
    """All the figures of one case; returns its row of the thickness table."""
    J2, dx, dy = J_squared_at(case)
    sheet = case["sheet"]
    when = rf"$\beta_p = {case['beta_p']}$, t = {SNAPSHOT_TIME}" + (f", {case['filter']} filter" if case["filter"] else "")

    # ---- the detected sheets, their maxima and the chosen sheet ----
    mask = quantile_mask(J2, SHEET_QUANTILE, SMOOTHING_RADIUS_CELLS, MIN_AREA_CELLS)
    maxima, labeled = get_max_coords(mask, J2)
    print(f"{name}: {len(maxima)} sheets above the {SHEET_QUANTILE} quantile")
    for figure, colour, points in (("sheets", "yellow", None), ("sheet_maxima", "white", maxima), ("chosen_sheet", "white", [sheet])):
        fig, ax = plt.subplots(figsize=(10, 8))
        show_field(ax, J2, dx, dy, label=r"$J^2$")
        outline_mask(ax, mask, dx, dy, color=colour)
        if points is not None:
            mark_cells(ax, points, dx, dy)
        if figure == "chosen_sheet":
            x, y = cell_centre(sheet, dx, dy)
            ax.axvline(x, color="yellow", alpha=0.25)
            ax.axhline(y, color="cyan", alpha=0.25)
        ax.set_title(r"$J^2$ and its current sheets, " + when)
        save_figure(fig, RESULTS, f"J2_{figure}_{name}")
        plt.show()

    # ---- the cut across the chosen sheet, along the steepest descent of J^2 ----
    direction = descent_direction(J2, sheet)
    s, profile = line_profile(J2, sheet, direction, case["cut_half_length"])
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(s * dx, profile)
    ax.axvline(0, linestyle="--")
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(r"$J^2$")
    ax.set_title(r"$J^2$ along the steepest descent, " + when, fontsize=9)
    save_figure(fig, RESULTS, f"J2_cut_{name}")
    plt.show()

    fig, ax = plt.subplots(figsize=(6, 6))
    zoom(ax, J2, sheet, CUT_ZOOM_HALF_BOX, dx, dy, label=r"$J^2$")
    x, y = cell_centre(sheet, dx, dy)
    ax.plot(x + s * direction[0] * dx, y + s * direction[1] * dy, color="yellow", linewidth=2)
    ax.scatter([x], [y], color="white", marker="x")
    ax.set_title("The cut", fontsize=10)
    save_figure(fig, RESULTS, f"J2_cut_zoom_{name}")
    plt.show()

    # ---- the five thicknesses, in cells ----
    thickness = {}
    region = labeled == labeled[sheet]
    thickness["method 1"] = thickness_m1(region) if labeled[sheet] else np.nan
    thickness["method 2"] = width_along(mask, sheet, direction)[0]
    k, values, parameters, thickness["method 3"] = fwhm_nearest_samples(J2, sheet, direction, FWHM_HALF_LENGTH_CELLS)
    thresholds = {"method 4": METHOD4_FRACTION_OF_MAX * J2.max(), "method 5": METHOD5_FRACTION_OF_PEAK * J2[sheet]}
    regions = {}
    for method, threshold in thresholds.items():
        regions[method] = grow_region(J2, sheet, threshold, case["region_smoothing"])
        thickness[method] = width_along(regions[method], sheet, direction)[0]
    if not labeled[sheet]:
        print(f"{name}: the chosen cell {sheet} is not inside a detected sheet; methods 1 and 2 are not defined there")
    for method, value in thickness.items():
        print(f"  {method}: {value:.2f} cells = {value * dx:.3f} d_p")

    # method 3: the Gaussian fit
    A, mu, sigma, B = parameters
    half_maximum = B + A / 2
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot(k * dx, values, label="Profile along the cut", color=LINE_COLOURS[0])
    ax.plot(k * dx, gaussian(k, *parameters), "--", label="Gaussian fit", color=LINE_COLOURS[1])
    for edge in (mu - thickness["method 3"] / 2, mu + thickness["method 3"] / 2):
        ax.axvline(edge * dx, linestyle="--", color="black")
    ax.axhline(half_maximum, linestyle="--", color=LINE_COLOURS[2], label=f"Half maximum = {half_maximum:.3f}")
    ax.scatter([mu * dx], [gaussian(mu, *parameters)], color="red", zorder=3, label="Peak")
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(r"$J^2$")
    ax.set_title(f"Method 3: FWHM = {thickness['method 3'] * dx:.3f} $d_p$, " + when, fontsize=9)
    ax.legend(loc="upper left", bbox_to_anchor=(1, 1))
    save_figure(fig, RESULTS, f"J2_fwhm_{name}")
    plt.show()

    # methods 4 and 5: the regions grown from the sheet
    for method, region_of_method in regions.items():
        fig, ax = plt.subplots(figsize=(6, 6))
        zoom(ax, J2, sheet, case["zoom_half_box"], dx, dy, label=r"$J^2$")
        outline_mask(ax, region_of_method, dx, dy, color="yellow", linewidth=2)
        ax.scatter([x], [y], color="white", marker="x", label="Maximum")
        ax.set_title(f"{method.capitalize()}: region above {thresholds[method]:.3g}", fontsize=10)
        ax.legend()
        save_figure(fig, RESULTS, f"J2_{method.replace(' ', '')}_region_{name}")
        plt.show()

    # all five on one cut
    s, profile = line_profile(J2, sheet, direction, case["comparison_half_length"])
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(s * dx, profile, label=r"$J^2$ profile", color="black")
    ax.axvline(0, linestyle="--", color="gray")
    for number, ((method, value), colour) in enumerate(zip(thickness.items(), LINE_COLOURS[1:])):
        half = value / 2 * dx
        ax.axvline(-half, linestyle="--", color=colour)
        ax.axvline(half, linestyle="--", color=colour)
        ax.annotate("", xy=(half, profile.max() * 0.8), xytext=(-half, profile.max() * 0.8),
                    arrowprops=dict(arrowstyle="<->", color=colour, linewidth=2))
        ax.text(0, profile.max() * (0.85 - 0.05 * number), f"{method}: {value * dx:.3f} $d_p$", color=colour, ha="center")
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(r"$J^2$")
    ax.set_title("The thickness of one sheet by five methods, " + when, fontsize=9)
    save_figure(fig, RESULTS, f"thickness_methods_{name}")
    plt.show()

    # the thresholds behind the methods, on the cut
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(s * dx, profile, label=r"$J^2$ profile", color="black")
    for level, label in ((np.quantile(J2, SHEET_QUANTILE), "Methods 1 and 2"), (half_maximum, "Method 3"),
                         (thresholds["method 4"], "Method 4"), (thresholds["method 5"], "Method 5")):
        ax.axhline(level, linestyle="--", label=label)
    ax.set_xlabel(r"Distance along the cut ($d_p$)")
    ax.set_ylabel(r"$J^2$")
    ax.set_title(r"Cut of $J^2$ with the thresholds, " + when, fontsize=9)
    ax.legend()
    save_figure(fig, RESULTS, f"J2_cut_thresholds_{name}")
    plt.show()

    row = dict(case=name, beta_p=case["beta_p"], filter=case["filter"] or "none", sheet_i=sheet[0], sheet_j=sheet[1])
    row.update({f"{method.replace(' ', '_')}_d_p": value * dx for method, value in thickness.items()})
    return row

In [ ]:
rows = [analyse_case(name, case) for name, case in CASES.items()]

with open(RESULTS / "thicknesses.csv", "w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print(f"saved {RESULTS / 'thicknesses.csv'}")